## Connecting to GPU

In [17]:
gpu_info = !nvidia-smi
gpu_info

['Wed Mar  4 17:47:59 2026       ',
 '+-----------------------------------------------------------------------------------------+',
 '| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |',
 '+-----------------------------------------+------------------------+----------------------+',
 '| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |',
 '| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |',
 '|                                         |                        |               MIG M. |',
 '|=========================================+========================+======================|',
 '|   0  NVIDIA GeForce RTX 2070      WDDM  |   00000000:01:00.0  On |                  N/A |',
 '| 33%   54C    P2             55W /  175W |    2630MiB /   8192MiB |     11%      Default |',
 '|                                         |                        |                  N/A |',
 '+-

## Installing Libraries

In [18]:
from IPython.display import Markdown, Audio
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, AutoProcessor, AutoModelForSpeechSeq2Seq
import torch, torchaudio
import getpass
import soundfile as sf
import accelerate

## Mount the Audio File

In [3]:
waveform, sample_rate = torchaudio.load("meeting_audio.wav")

In [4]:
waveform


tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [5]:
waveform.shape

torch.Size([2, 434176])

In [6]:
sample_rate

48000

In [7]:
# Convert to mono
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0)  # shape [N]

# Resample to 16kHz if needed
if sample_rate != 16000:
    waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(waveform)
    sample_rate = 16000

### Load Whisper Medium Model

In [8]:
processor = AutoProcessor.from_pretrained("./whisper_medium_en", local_files_only=True)

In [9]:
feature_extractor = processor.feature_extractor
tokenizer = processor.tokenizer

In [ ]:
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    "./whisper_medium_en",
    device_map="auto",           
    dtype=torch.float16,
    max_memory={0: "4GiB"},   
    local_files_only=True
)

In [11]:
from transformers import pipeline
 
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
    dtype=torch.float16
)

Device set to use cuda:0


In [12]:
result = pipe(waveform)
transcript = result['text']

`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 357, 366, 438, 532, 685, 705, 796, 930, 1058, 1220, 1267, 1279, 1303, 1343, 1377, 1391, 1635, 1782, 1875, 2162, 2361, 2488, 3467, 4008, 4211, 4600, 4808, 5299, 5855, 6329, 7203, 9609, 9959, 10563, 10786, 11420, 11709, 11907, 13163, 13697, 13700, 14808, 15306, 16410, 16791, 17992, 19203, 19510, 20724, 22305, 22935, 27007, 30109, 30420, 33409, 34949, 40283, 40493, 40549, 47282, 49146, 50257, 50357, 50358, 50359, 50360, 50361], 'begin_suppress_tokens': [220, 50256]}

In [13]:
Markdown(transcript)

 Hello, how are you? We are planning to conduct our next meeting on 24th February at 4 pm.

In [15]:
import gc
del model
del processor
del pipe
del waveform  # or waveform_updated if you used that

# 2. Run garbage collector
gc.collect()

# 3. Clear cached memory on GPU
torch.cuda.empty_cache()